# Milestone 2: Model Architectures, Two-Phase Training & Clinical Calibration

This runnable walkthrough demonstrates the Milestone 2 modeling and calibration stack in **MRITumorDetection-CNN**:
1. **Pure CNN Tensor Trace & Backbone Comparison**: Intermediate shapes (`forward_features -> pool -> head -> logits`) and parameter comparison across `SimpleCNN`, `TimmClassifier("efficientnet_b0")`, and `TimmClassifier("resnet50")` (with `pretrained=False`).
2. **Two-Phase Backbone Freezing**: Verifying `freeze_backbone()` / `unfreeze_backbone()` trainable parameter counts and `BatchNorm` `.eval()` locking even under `model.train()`.
3. **Imbalance Losses & 4-Epoch `Trainer` Run**: Class weighting (`compute_class_weights`), `FocalLoss` / `build_loss_fn`, linear warmup -> cosine decay LR schedule, and loss/LR curves.
4. **Post-Hoc Clinical Calibration**: Stage 1 sensitivity-constrained thresholding (`find_sensitivity_threshold`, `target_sensitivity=0.98`) and Stage 2 L-BFGS `TemperatureScaler` + confidence abstention (`apply_temperature_and_abstention`, `uncertainty_threshold=0.70`).

## 1. Imports & Reproducibility Setup

In [ ]:
import sys
import tempfile
from pathlib import Path

# Ensure src/ is importable if the notebook is run without editable install
repo_root = Path.cwd().resolve()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from mri_tumor.models import (
    SimpleCNN,
    TimmClassifier,
    build_model_from_config,
)
from mri_tumor.training import (
    compute_class_weights,
    FocalLoss,
    build_loss_fn,
    find_sensitivity_threshold,
    TemperatureScaler,
    apply_temperature_and_abstention,
    Trainer,
    set_seed,
)

set_seed(42)
print(f"PyTorch: {torch.__version__} | Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. CNN Architecture Instantiation, Exact Tensor Shape Trace & Parameter Comparison

We inspect:
- Stage 1 screening classifier: `TimmClassifier("efficientnet_b0", num_classes=2, pretrained=False)`
- Stage 2 subtyping CNNs: `SimpleCNN(num_classes=3)`, `TimmClassifier("efficientnet_b0", num_classes=3, pretrained=False)`, and `TimmClassifier("resnet50", num_classes=3, pretrained=False)`
- Config-driven instantiation via `build_model_from_config`

In [ ]:
x_dummy = torch.randn(2, 3, 224, 224)

# Stage 1 screening CNN
stage1_model = TimmClassifier("efficientnet_b0", num_classes=2, pretrained=False)
stage1_model.eval()
with torch.no_grad():
    s1_feat = stage1_model.forward_features(x_dummy)
    s1_logits = stage1_model(x_dummy)

print("=== Stage 1: TimmClassifier('efficientnet_b0', num_classes=2) ===")
print(f"  input            : {tuple(x_dummy.shape)}")
print(f"  forward_features : {tuple(s1_feat.shape)}")
print(f"  logits           : {tuple(s1_logits.shape)}")
print(f"  total params     : {sum(p.numel() for p in stage1_model.parameters()):,}")

# Stage 2 CNN comparisons (SimpleCNN baseline vs. EfficientNet-B0 primary vs. ResNet-50 comparison)
model_simple = SimpleCNN(num_classes=3, dropout=0.3)
model_effnet = TimmClassifier("efficientnet_b0", num_classes=3, pretrained=False, dropout=0.3)
model_resnet = TimmClassifier("resnet50", num_classes=3, pretrained=False, dropout=0.3)

def trace_cnn(model: nn.Module, x: torch.Tensor, label: str) -> tuple[int, int]:
    model.eval()
    with torch.no_grad():
        feat = model.forward_features(x)
        pooled = model.pool(feat)
        logits = model.head(pooled)

    print(f"\n=== {label} ===")
    print(f"  input            : {tuple(x.shape)}")
    print(f"  forward_features : {tuple(feat.shape)}")
    print(f"  pool             : {tuple(pooled.shape)}")
    print(f"  logits           : {tuple(logits.shape)}")
    print(f"  target_layer     : {type(model.get_target_layer()).__name__}")
    return feat.shape[2], feat.shape[3]

hw_simple = trace_cnn(model_simple, x_dummy, "Stage 2: SimpleCNN (num_classes=3)")
hw_effnet = trace_cnn(model_effnet, x_dummy, "Stage 2: TimmClassifier('efficientnet_b0', num_classes=3)")
hw_resnet = trace_cnn(model_resnet, x_dummy, "Stage 2: TimmClassifier('resnet50', num_classes=3)")

In [ ]:
def count_module_params(m: nn.Module) -> dict[str, int]:
    return {
        "backbone": sum(p.numel() for p in m.backbone.parameters()),
        "head": sum(p.numel() for p in m.head.parameters()),
        "total": sum(p.numel() for p in m.parameters()),
    }

cnn_models = {
    "SimpleCNN": model_simple,
    "EfficientNet-B0": model_effnet,
    "ResNet-50": model_resnet,
}
param_summary = {name: count_module_params(m) for name, m in cnn_models.items()}

print("Parameter breakdown (3-class Stage 2 CNNs):")
for name, counts in param_summary.items():
    print(
        f"  {name:16s}: backbone={counts['backbone']:>10,} | "
        f"head={counts['head']:>6,} | total={counts['total']:>10,}"
    )

# Config-based build checks
cfg_effnet = build_model_from_config({
    "model": {
        "arch": "cnn",
        "backbone": "efficientnet_b0",
        "num_classes": 3,
        "pretrained": False,
        "dropout": 0.3,
    },
})
cfg_simple = build_model_from_config({
    "model": {
        "arch": "simple_cnn",
        "num_classes": 3,
        "dropout": 0.3,
    },
})
print(f"\nbuild_model_from_config ('cnn'): {type(cfg_effnet).__name__} (backbone={cfg_effnet.backbone_name})")
print(f"build_model_from_config ('simple_cnn'): {type(cfg_simple).__name__} (feature_dim={cfg_simple.feature_dim})")

# Plot parameter comparison & feature-map channel/spatial footprints
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(param_summary.keys())
totals_m = [param_summary[n]["total"] / 1e6 for n in names]
colors = ["#2ca02c", "#1f77b4", "#d62728"]
bars = axes[0].bar(names, totals_m, color=colors)
axes[0].set_ylabel("Total Parameters (Millions)")
axes[0].set_title("Stage 2 CNN Capacity: SimpleCNN vs. EfficientNet-B0 vs. ResNet-50")
axes[0].grid(axis="y", alpha=0.3)
for bar, val in zip(bars, totals_m):
    axes[0].text(bar.get_x() + bar.get_width() / 2.0, val + 0.4, f"{val:.2f}M", ha="center", va="bottom", fontsize=9)

feat_dims = [model_simple.feature_dim, model_effnet.feature_dim, model_resnet.feature_dim]
grid_labels = [f"{n}\n({hw[0]}x{hw[1]})" for n, hw in zip(names, [hw_simple, hw_effnet, hw_resnet])]
bars2 = axes[1].bar(grid_labels, feat_dims, color=colors)
axes[1].set_ylabel("Pooled Feature Dimension (C)")
axes[1].set_title("Final Conv Channel Width & Spatial Grid Size")
axes[1].grid(axis="y", alpha=0.3)
for bar, val in zip(bars2, feat_dims):
    axes[1].text(bar.get_x() + bar.get_width() / 2.0, val + 40, str(val), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Two-Phase Backbone Freezing & BatchNorm Eval Locking

`_BackboneControlMixin` guarantees that during Phase 1 (`freeze_backbone()`), **both**:
1. Backbone weights have `requires_grad = False` (no gradient updates / no gradient shock).
2. `backbone.training` stays `False` even when `model.train()` is called by `Trainer` (so pretrained `BatchNorm2d` running mean/variance do not drift).

In [ ]:
def summarize_freeze_state(model: nn.Module, tag: str) -> None:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    bn_training_flags = [
        m.training for m in model.backbone.modules() if isinstance(m, nn.BatchNorm2d)
    ]
    print(
        f"[{tag}] model.training={model.training} | "
        f"backbone.training={model.backbone.training} | "
        f"any_backbone_BN_train={any(bn_training_flags)} | "
        f"trainable={trainable:,} | frozen={frozen:,}"
    )

freeze_demo = TimmClassifier(
    backbone_name="efficientnet_b0",
    num_classes=3,
    pretrained=False,
)

freeze_demo.train()
summarize_freeze_state(freeze_demo, "Initial (unfrozen)")

freeze_demo.freeze_backbone()
summarize_freeze_state(freeze_demo, "After freeze_backbone()")

# Simulate Trainer calling model.train() at the start of an epoch during Phase 1
freeze_demo.train()
summarize_freeze_state(freeze_demo, "After model.train() while frozen")
assert freeze_demo.backbone.training is False, "Backbone BN must stay in eval mode while frozen!"

freeze_demo.unfreeze_backbone()
summarize_freeze_state(freeze_demo, "After unfreeze_backbone()")
assert freeze_demo.backbone.training is True

## 4. Class-Imbalance Losses & 4-Epoch Two-Phase `Trainer` Run

We compute balanced class weights (`compute_class_weights`), compare `build_loss_fn("weighted_ce", ...)` and `FocalLoss`, and run a fast 4-epoch synthetic training loop with `Trainer(..., epochs=4, freeze_backbone_epochs=2, warmup_epochs=1)` on a lightweight `SimpleCNN` model.

In [ ]:
# Imbalanced 3-class synthetic dataset (e.g., glioma=60, meningioma=30, pituitary=18)
set_seed(42)
train_labels = torch.tensor([0] * 60 + [1] * 30 + [2] * 18, dtype=torch.long)
val_labels = torch.tensor([0] * 16 + [1] * 8 + [2] * 6, dtype=torch.long)

class_weights = compute_class_weights(train_labels, num_classes=3)
print(f"Balanced class_weights (mean={class_weights.mean().item():.4f}): {class_weights.tolist()}")

weighted_ce = build_loss_fn("weighted_ce", class_weights=class_weights, label_smoothing=0.05)
focal_fn = FocalLoss(gamma=2.0, alpha=class_weights, label_smoothing=0.05)
focal_via_factory = build_loss_fn("focal", class_weights=class_weights, label_smoothing=0.05, focal_gamma=2.0)

sample_logits = torch.randn(8, 3)
sample_targets = torch.tensor([0, 1, 2, 0, 1, 2, 0, 2])
print(f"Weighted CE loss : {weighted_ce(sample_logits, sample_targets).item():.4f}")
print(f"FocalLoss        : {focal_fn(sample_logits, sample_targets).item():.4f}")
assert torch.isclose(focal_fn(sample_logits, sample_targets), focal_via_factory(sample_logits, sample_targets))

# Build separable synthetic 64x64 MRI-like blobs so training converges quickly on CPU
def make_synthetic_images(labels: torch.Tensor, img_size: int = 64) -> torch.Tensor:
    n = len(labels)
    imgs = torch.randn(n, 3, img_size, img_size) * 0.15
    for i, lbl in enumerate(labels.tolist()):
        # Place bright lesion signatures in superior (0), middle (1), or inferior (2) bands
        r0 = 8 + lbl * 18
        imgs[i, :, r0 : r0 + 12, 24:40] += 1.5
    return imgs

train_ds = TensorDataset(make_synthetic_images(train_labels), train_labels)
val_ds = TensorDataset(make_synthetic_images(val_labels), val_labels)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

fast_model = SimpleCNN(num_classes=3, dropout=0.2)

with tempfile.TemporaryDirectory() as tmp_ckpt_dir:
    trainer = Trainer(
        model=fast_model,
        criterion=weighted_ce,
        device="cpu",
        epochs=4,
        lr=1e-3,
        warmup_epochs=1,
        freeze_backbone_epochs=2,
        early_stopping_patience=5,
        checkpoint_dir=tmp_ckpt_dir,
        use_amp=False,
    )
    history = trainer.fit(train_loader, val_loader)

for ep in range(len(history.train_loss)):
    phase = "Phase 1 (Frozen)" if ep < 2 else "Phase 2 (Unfrozen)"
    print(
        f"Epoch {ep} [{phase}] | lr={history.lr[ep]:.6f} | "
        f"train_loss={history.train_loss[ep]:.4f} | val_loss={history.val_loss[ep]:.4f} | "
        f"val_acc={history.val_acc[ep]:.3f}"
    )
print(f"Best epoch: {history.best_epoch} (val_loss={history.best_val_loss:.4f})")

# Plot training/validation curves & learning-rate schedule
epochs_x = np.arange(len(history.train_loss))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_x, history.train_loss, marker="o", label="Train Loss")
axes[0].plot(epochs_x, history.val_loss, marker="s", label="Val Loss")
axes[0].axvline(1.5, color="gray", linestyle="--", label="Backbone Unfreeze (Epoch 2)")
axes[0].set_xticks(epochs_x)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Weighted CE Loss")
axes[0].set_title("4-Epoch Synthetic Two-Phase Training")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, history.lr, marker="^", color="#9467bd", label="Learning Rate")
axes[1].axvline(1.5, color="gray", linestyle="--", label="Backbone Unfreeze (Epoch 2)")
axes[1].set_xticks(epochs_x)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("LR")
axes[1].set_title("Linear Warmup -> Cosine Annealing Schedule")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Post-Hoc Calibration: Stage 1 Sensitivity Thresholding & Stage 2 Temperature Scaling + Abstention

1. **Stage 1 (`find_sensitivity_threshold`)**: Finds the highest probability cutoff maximizing specificity subject to `Sensitivity >= 0.98`.
2. **Stage 2 (`TemperatureScaler` + `apply_temperature_and_abstention`)**: Fits scalar $T = \exp(\text{log\_temperature})$ via L-BFGS NLL minimization on overconfident logits, then abstains whenever $\max_c p_c < 0.70$.

In [ ]:
# --- Stage 1: Sensitivity-Constrained Threshold Search ---
rng = np.random.default_rng(42)
n_neg, n_pos = 200, 200
s1_targets = np.array([0] * n_neg + [1] * n_pos, dtype=np.int64)
neg_probs = np.clip(rng.beta(2.0, 6.0, size=n_neg), 0.01, 0.95)
pos_probs = np.clip(rng.beta(6.0, 1.4, size=n_pos), 0.05, 0.99)
s1_probs = np.concatenate([neg_probs, pos_probs])

thr_res = find_sensitivity_threshold(s1_probs, s1_targets, target_sensitivity=0.98)
print("=== Stage 1 Clinical Sensitivity Calibration ===")
print(
    f"  threshold={thr_res.threshold:.4f} | sensitivity={thr_res.sensitivity:.4f} | "
    f"specificity={thr_res.specificity:.4f} | precision={thr_res.precision:.4f} | "
    f"achieved_target={thr_res.achieved_target}"
)

# --- Stage 2: Temperature Scaling & Confidence Abstention ---
# Simulate overconfident logits (high temperature distortion ~ 2.4)
n_val = 300
s2_labels = rng.integers(0, 3, size=n_val)
true_temp = 2.4
raw_logits = rng.normal(loc=0.0, scale=1.2, size=(n_val, 3)).astype(np.float32)
for i, y in enumerate(s2_labels):
    raw_logits[i, y] += 2.2
overconfident_logits = raw_logits * true_temp

nll_fn = nn.CrossEntropyLoss()
logits_t = torch.tensor(overconfident_logits, dtype=torch.float32)
labels_t = torch.tensor(s2_labels, dtype=torch.long)
nll_before = nll_fn(logits_t, labels_t).item()

scaler = TemperatureScaler(init_temperature=1.0)
fitted_temp = scaler.fit(overconfident_logits, s2_labels, max_iter=50, lr=0.05)
with torch.no_grad():
    nll_after = nll_fn(scaler(logits_t), labels_t).item()

res_before = apply_temperature_and_abstention(
    overconfident_logits, temperature=1.0, uncertainty_threshold=0.70
)
res_after = apply_temperature_and_abstention(
    overconfident_logits, temperature=fitted_temp, uncertainty_threshold=0.70
)

print("\n=== Stage 2 Temperature Scaling & Abstention ===")
print(f"  Fitted Temperature T : {fitted_temp:.4f}")
print(f"  Validation NLL       : {nll_before:.4f} -> {nll_after:.4f}")
print(
    f"  Mean Confidence      : {res_before['confidences'].mean():.4f} -> "
    f"{res_after['confidences'].mean():.4f}"
)
print(
    f"  Abstention Rate (<0.70): {res_before['abstain_mask'].mean()*100:.1f}% -> "
    f"{res_after['abstain_mask'].mean()*100:.1f}%"
)
abstain_examples = [lbl for lbl, mask in zip(res_after["labels"], res_after["abstain_mask"]) if mask]
if abstain_examples:
    print(f"  Sample abstained label : {abstain_examples[0]!r}")

# Visualize Stage 1 sensitivity/specificity trade-off and Stage 2 calibration effect
grid_t = np.linspace(0.05, 0.95, 100)
sens_curve = [np.mean(s1_probs[s1_targets == 1] >= t) for t in grid_t]
spec_curve = [np.mean(s1_probs[s1_targets == 0] < t) for t in grid_t]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(grid_t, sens_curve, label="Sensitivity", color="#d62728")
axes[0].plot(grid_t, spec_curve, label="Specificity", color="#1f77b4")
axes[0].axvline(thr_res.threshold, color="black", linestyle="--", label=f"Calibrated tau={thr_res.threshold:.3f}")
axes[0].axhline(0.98, color="gray", linestyle=":", label="Target Sens=0.98")
axes[0].set_xlabel("Decision Threshold")
axes[0].set_ylabel("Metric Value")
axes[0].set_title("Stage 1: Sensitivity-Constrained Thresholding")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].hist(res_before["confidences"], bins=20, alpha=0.55, label="Uncalibrated (T=1.0)", color="#ff7f0e")
axes[1].hist(res_after["confidences"], bins=20, alpha=0.65, label=f"Calibrated (T={fitted_temp:.2f})", color="#2ca02c")
axes[1].axvline(0.70, color="red", linestyle="--", label="Abstain Cutoff = 0.70")
axes[1].set_xlabel("Max Softmax Probability")
axes[1].set_ylabel("Sample Count")
axes[1].set_title("Stage 2: Confidence Softening & Abstention")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()